# Temporal Risk Evaluation using Bayesian Inference
**Author:** Jake (Team 6)  
**Objective:** To address the issue of mixed, noisy predictions (e.g., Cross, NoCross, Cross) across sequential video frames. 

Instead of relying on simple rolling averages (which introduce latency) or static thresholds, this notebook implements **Recursive Bayesian Updating** (as taught in CS3264). By treating our base classifier's frame-by-frame output as noisy observations, we continuously update our prior belief of the pedestrian's intention using Bayes' Theorem. 

Furthermore, we implement a **Distance-Weighted Dynamic Threshold** utilizing the YOLO bounding box height (a proxy for depth) to ensure high sensitivity when pedestrians are close to the vehicle.

In [1]:
import pandas as pd
import math

print("Loading advanced predictions...")
df = pd.read_csv('advanced_predictions.csv')

# Sort values properly to ensure temporal sequence
df = df.sort_values(['video_id', 'pedestrian_id', 'frame_id']).reset_index(drop=True)
df['track_id'] = df['video_id'].astype(str) + "_" + df['pedestrian_id'].astype(str)

# **** OPTIMIZED KINEMATIC BAYESIAN PARAMETERS ****
PRIOR_PROB = 0.5
DECAY_RISING = 0.98      # Accumulate risk faster
DECAY_FALLING = 0.85     # Slower decay to maintain memory of risk
TREND_WINDOW = 2         # Wait 2 frames to confirm a trend
BASE_THRESHOLD = 0.30    # Lowered slightly to catch more crossers

# FORGETTING FACTOR
FORGETTING_FACTOR = 0.95 # Retain 95% of belief per frame, drift towards 0.5

# EMERGENCY BYPASS CONSTANTS
EMERGENCY_BBOX_HEIGHT = 0.25  # Pedestrian takes up >25% of vertical screen
EMERGENCY_PROB = 0.30         # Raw RF probability required for instant trigger

def prob_to_log_odds(p):
    p = max(1e-5, min(1 - 1e-5, p))
    return math.log(p / (1 - p))

def log_odds_to_prob(lo):
    return 1 / (1 + math.exp(-lo))

results = []

print("Running Bayesian Updating...")
for _, group in df.groupby('track_id'):
    current_prob = PRIOR_PROB
    recent_probs = []
    
    for _, row in group.iterrows():
        rf_prob = row['advanced_prediction']
        bbox_h = row['bbox_height']
        
        # 1. Prior Forgetting: Decay previous frame's belief toward maximum uncertainty (0.5)
        current_prob = 0.5 + (current_prob - 0.5) * FORGETTING_FACTOR

        # 2. Asymmetric Decay Direction Fix: Shrink RF observation toward 0.5
        if rf_prob >= 0.5:
            rf_prob = 0.5 + (rf_prob - 0.5) * DECAY_RISING
        else:
            rf_prob = 0.5 + (rf_prob - 0.5) * DECAY_FALLING
            
        # Bayesian update via log odds
        prior_lo = prob_to_log_odds(current_prob)
        obs_lo = prob_to_log_odds(rf_prob)
        
        new_lo = prior_lo + obs_lo
        current_prob = log_odds_to_prob(new_lo)
        recent_probs.append(current_prob)
        
        # Maintain short trend window
        if len(recent_probs) > TREND_WINDOW:
            recent_probs.pop(0)
        
        # AGGRESSIVE DYNAMIC DISTANCE-WEIGHTED THRESHOLD
        if bbox_h > 0.2:
            dynamic_threshold = 0.35  
        elif bbox_h < 0.1:
            dynamic_threshold = 0.60  
        else:
            dynamic_threshold = BASE_THRESHOLD
        
        # Standard Trigger logic based on trend window 
        trigger_warning = 0
        if len(recent_probs) == TREND_WINDOW and all(p >= dynamic_threshold for p in recent_probs):
            trigger_warning = 1
            
        # If the pedestrian is dangerously close and the model detects intent,
        # trigger instantly. Do not wait for the 2 frame trend window.
        if bbox_h >= EMERGENCY_BBOX_HEIGHT and row['advanced_prediction'] >= EMERGENCY_PROB:
            trigger_warning = 1
            
        results.append({
            'video_id': row['video_id'],
            'pedestrian_id': row['pedestrian_id'],
            'frame_id': row['frame_id'],
            'bayesian_risk_prob': current_prob,
            'dynamic_threshold_used': dynamic_threshold,
            'warning_triggered': trigger_warning
        })

# Save temporal results
results_df = pd.DataFrame(results)
final_df = pd.merge(df, results_df, on=['video_id', 'pedestrian_id', 'frame_id'])
final_df.to_csv('temporal_evaluation_results.csv', index=False)
print("Temporal processing complete. Saved to temporal_evaluation_results.csv")

Loading advanced predictions...
Running Bayesian Updating...
Temporal processing complete. Saved to temporal_evaluation_results.csv


In [2]:
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np
import pandas as pd

df_evaluated = pd.read_csv('temporal_evaluation_results.csv')

if 'crossing_actual' in df_evaluated.columns and 'crossing_shifted' in df_evaluated.columns:
    
    y_true_shifted = df_evaluated['crossing_shifted']
    y_prob_bayesian = df_evaluated['bayesian_risk_prob']
    y_pred_alert = df_evaluated['warning_triggered']
    
    print("\n" + "="*40)
    print("BAYESIAN TEMPORAL MODEL PERFORMANCE")
    print("="*40)
    
    temporal_auroc = roc_auc_score(y_true_shifted, y_prob_bayesian)
    print(f"Temporal AUROC: {temporal_auroc:.4f}")
    
    print("\n" + "="*40)
    print("AUTONOMOUS VEHICLE SAFETY METRICS")
    print("="*40)

    lead_times_seconds = []
    false_alarms = 0
    
    successful_warnings = 0
    impossible_scenarios = 0
    true_model_misses = 0
    
    total_crossers = 0
    total_non_crossers = 0

    for (vid, ped_id), track_df in df_evaluated.groupby(['video_id', 'pedestrian_id']):
        is_crosser = (track_df['crossing_actual'] == 1).any()
        triggered_alert = (track_df['warning_triggered'] == 1).any()

        if is_crosser:
            total_crossers += 1
            actual_cross_frame = track_df[track_df['crossing_actual'] == 1]['frame_id'].min()
            first_visible_frame = track_df['frame_id'].min()
            
            # How many frames was the pedestrian on camera before crossing?
            visibility_frames = actual_cross_frame - first_visible_frame
            
            if triggered_alert:
                trigger_frame = track_df[track_df['warning_triggered'] == 1]['frame_id'].min()
                
                if trigger_frame <= actual_cross_frame:
                    frames_saved = actual_cross_frame - trigger_frame
                    seconds_saved = frames_saved / 30.0
                    lead_times_seconds.append(seconds_saved)
                    successful_warnings += 1
                else:
                    # Triggered late
                    if visibility_frames < 30: # Less than 1 second of visibility
                        impossible_scenarios += 1
                    else:
                        true_model_misses += 1
            else:
                # Never triggered
                if visibility_frames < 30:
                    impossible_scenarios += 1
                else:
                    true_model_misses += 1
        else:
            total_non_crossers += 1
            if triggered_alert:
                false_alarms += 1

    print(f"Total Pedestrians Evaluated: {total_crossers + total_non_crossers}")
    print(f"  Pedestrians who crossed: {total_crossers}")
    print(f"  Pedestrians who never crossed: {total_non_crossers}\n")

    print("=== EARLY WARNING LEAD TIME ===")
    if lead_times_seconds:
        print(f"Average Warning Time: {np.mean(lead_times_seconds):.2f} seconds")
        print(f"Median Warning Time:  {np.median(lead_times_seconds):.2f} seconds")
        print(f"Maximum Warning Time: {np.max(lead_times_seconds):.2f} seconds")
        print(f"Minimum Warning Time: {np.min(lead_times_seconds):.2f} seconds")
    else:
        print("Average Warning Time: 0.00 seconds")

    print("\n=== SAFETY RELIABILITY BREAKDOWN ===")
    success_rate = (successful_warnings / total_crossers) * 100 if total_crossers else 0
    impossible_rate = (impossible_scenarios / total_crossers) * 100 if total_crossers else 0
    true_miss_rate = (true_model_misses / total_crossers) * 100 if total_crossers else 0
    false_alarm_rate = (false_alarms / total_non_crossers) * 100 if total_non_crossers else 0
    
    print(f"Successful Early Warnings:      {successful_warnings} / {total_crossers} ({success_rate:.1f}%)")
    print(f"Occlusion-Limited Scenarios: {impossible_scenarios} / {total_crossers} ({impossible_rate:.1f}%)")
    print(f"True Model Misses/Late:         {true_model_misses} / {total_crossers} ({true_miss_rate:.1f}%)")
    print(f"False Alarms (Non-crossers):    {false_alarms} / {total_non_crossers} ({false_alarm_rate:.1f}%)")

else:
    print("Warning: Check column names.")


BAYESIAN TEMPORAL MODEL PERFORMANCE
Temporal AUROC: 0.9858

AUTONOMOUS VEHICLE SAFETY METRICS
Total Pedestrians Evaluated: 651
  Pedestrians who crossed: 497
  Pedestrians who never crossed: 154

=== EARLY WARNING LEAD TIME ===
Average Warning Time: 2.33 seconds
Median Warning Time:  2.10 seconds
Maximum Warning Time: 12.67 seconds
Minimum Warning Time: 0.00 seconds

=== SAFETY RELIABILITY BREAKDOWN ===
Successful Early Warnings:      300 / 497 (60.4%)
Occlusion-Limited Scenarios: 194 / 497 (39.0%)
True Model Misses/Late:         3 / 497 (0.6%)
False Alarms (Non-crossers):    40 / 154 (26.0%)
